# Step 5: inspect an identity-continuity failure

This notebook reuses your saved controlled-experiment predictions. No new model inference or parameter changes occur. A CPU runtime is enough. It creates one detailed clip and one context clip from MOT17-02.

The candidate involves annotated pedestrian G32. Saved box-overlap records show full ByteTrack changing T26 to T36 and back to T26, while high_only retains T26. This is a stronger candidate than an unannotated distant detection. The clip still needs visual inspection: these are trajectory-overlap observations, not independently replayed TrackEval IDSW events or proof of a particular association mechanism.

Run all cells in order. Reuse the same Colab session if possible. If prompted, upload ByteTrack_official_controlled_comparison.zip. The notebook reuses original images when available; a new runtime without the dataset downloads approximately 5.5 GB into Colab.

The code and selection have been checked locally against your saved predictions. Rendering still needs execution in Colab. Follow your course's rules on AI-assisted work and write assessed observations yourself.


## 1. Load the saved comparison
If Colab still has ByteTrack_official_controlled_comparison.zip, it will use it. Otherwise this cell opens an upload button. Upload that ZIP from your Mac. The notebook copies only the files needed for the selected scene; your original experiment files remain unchanged.


In [ ]:
from pathlib import Path
import json, csv, zipfile, hashlib, urllib.request, subprocess, shutil, math
import numpy as np
import pandas as pd
import cv2
from IPython.display import Video, Image, display
from google.colab import files
ROOT = Path('/content/bytetrack_identity_review')
ROOT.mkdir(exist_ok=True)
SOURCE_ZIP = Path('/content/ByteTrack_official_controlled_comparison.zip')
if not SOURCE_ZIP.is_file():
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith('.zip')]
    assert len(candidates) == 1, 'Upload the single controlled-comparison ZIP.'
    SOURCE_ZIP = Path('/content') / Path(candidates[0]).name
    SOURCE_ZIP.write_bytes(uploaded[candidates[0]])
SEQUENCES = ['MOT17-02-FRCNN']
needed = ['split_info.json','environment.json','comparison.csv','cache_info.json']
for seq in SEQUENCES:
    needed += [f'gt/{seq}/gt/gt.txt', f'gt/{seq}/seqinfo.ini']
    needed += [f'trackers/{v}/data/{seq}.txt' for v in ['high_only','full_bytetrack']]
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    assert all(name in archive.namelist() for name in needed), 'This is not the expected controlled-comparison archive.'
    for name in needed:
        destination = ROOT / name
        destination.parent.mkdir(parents=True,exist_ok=True)
        destination.write_bytes(archive.read(name))
info_by_seq = json.loads((ROOT/'split_info.json').read_text())
print('Loaded saved results for:', ', '.join(SEQUENCES))


## 2. Reuse or extract the original images
The validation-to-original mapping comes from your saved split_info.json. The notebook verifies file existence and original image dimensions. If the prior detector-cache signature is available in the source ZIP, it also checks the exact JPEG-frame hashes against that signature.


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path,'rb') as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def image_frames(info):
    return range(info['first_original_frame'],info['original_frames']+1)

image_folders = {}
missing = []
for seq in SEQUENCES:
    info = info_by_seq[seq]
    candidates = [Path('/content/bytetrack_official_controlled/images')/seq,
                  Path('/content/bytetrack_mot17_halfval/images')/seq,
                  Path('/content/bytetrack_visual_review/images')/seq,
                  ROOT/'images'/seq]
    folder = next((p for p in candidates if all((p/f'{i:06d}.jpg').is_file() for i in image_frames(info))),None)
    if folder is None: missing.append(seq)
    else: image_folders[seq] = folder
if missing:
    archive_path = Path('/content/MOT17.zip')
    if not archive_path.is_file():
        partial = archive_path.with_suffix('.zip.part')
        last = [-10]
        def progress(block,size,total):
            if total>0:
                percent = min(100,int(block*size*100/total))
                if percent >= last[0]+10:
                    print('MOT17 download:',percent,'%');last[0]=percent
        urllib.request.urlretrieve('https://motchallenge.net/data/MOT17.zip',str(partial),reporthook=progress)
        assert zipfile.is_zipfile(partial), 'Download failed; check the official MOT17 page.'
        partial.rename(archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        for seq in missing:
            ini = [n for n in names if n.endswith(f'train/{seq}/seqinfo.ini')]
            assert len(ini)==1
            prefix = ini[0][:-len('seqinfo.ini')]
            folder = ROOT/'images'/seq
            folder.mkdir(parents=True,exist_ok=True)
            for frame in image_frames(info_by_seq[seq]):
                (folder/f'{frame:06d}.jpg').write_bytes(archive.read(prefix+f'img1/{frame:06d}.jpg'))
            image_folders[seq]=folder
checks = {}
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    for seq in SEQUENCES:
        info = info_by_seq[seq]
        name = f'cache/{seq}.npz'
        assert name in archive.namelist(), 'Expected detector cache missing; use the complete comparison ZIP.'
        cache_path = ROOT/f'{seq}_detector_cache.npz'
        cache_path.write_bytes(archive.read(name))
        with np.load(cache_path,allow_pickle=False) as cached:
            signature = json.loads(str(cached['signature'].item()))
        digest = hashlib.sha256()
        for frame in image_frames(info):
            p = image_folders[seq]/f'{frame:06d}.jpg'
            digest.update(p.name.encode())
            digest.update(bytes.fromhex(sha256_file(p)))
        assert digest.hexdigest()==signature['image_sha256'], f'{seq}: source image bytes differ from the detector run.'
        example = cv2.imread(str(image_folders[seq]/f"{info['first_original_frame']:06d}.jpg"))
        assert example is not None and example.shape[:2]==(info['height'],info['width'])
        checks[seq]=dict(image_hash_matches_detector_run=True,dimensions_match=True)
(ROOT/'image_checks.json').write_text(json.dumps(checks,indent=2))
print('Original images match the detector-run image hashes for the selected scene.')


## 3. Check the selected pedestrian and ID timeline
This uses scored pedestrian G32, so an absent annotation is not the reason for the candidate. For each frame, find each variant's best-overlapping output on G32. The table is a visual-diagnostic association based on box overlap, not the evaluator's full temporal matching algorithm.

Expect full ByteTrack T26 through original frame 351, T36 from 352 through 362, then T26 again from 363 onward. Expect high_only T26 throughout the displayed range. A stable number alone is insufficient if the box follows the wrong person; verify the person in the video.


In [ ]:
"""Screen saved predictions for visual inspection; not an evaluation metric."""
import numpy as np

def load_rows(path, columns):
    return np.loadtxt(path, delimiter=',', ndmin=2) if path.stat().st_size else np.empty((0, columns))

def overlaps(a, b, use_ioa=False):
    a = np.asarray(a).reshape(-1, 4)
    b = np.asarray(b).reshape(-1, 4)
    if not len(a) or not len(b):
        return np.zeros((len(a), len(b)))
    a2, b2 = a[:, :2] + a[:, 2:], b[:, :2] + b[:, 2:]
    lo = np.maximum(a[:, None, :2], b[None, :, :2])
    hi = np.minimum(a2[:, None, :], b2[None, :, :])
    inter = np.maximum(hi - lo, 0).prod(axis=2)
    area_a = a[:, 2:].prod(axis=1)[:, None]
    if use_ioa:
        denominator = np.broadcast_to(area_a, inter.shape)
    else:
        denominator = area_a + b[:, 2:].prod(axis=1)[None, :] - inter
    return inter / np.maximum(denominator, 1e-9)

def max_overlap(a, b, use_ioa=False):
    values = overlaps(a, b, use_ioa)
    return values.max(axis=1) if len(b) else np.zeros(len(a))

def screen_candidates(root, sequences, info_by_seq):
    events = []
    for seq in sequences:
        gt = load_rows(root/'gt'/seq/'gt/gt.txt', 9)
        full = load_rows(root/'trackers/full_bytetrack/data'/f'{seq}.txt', 10)
        high = load_rows(root/'trackers/high_only/data'/f'{seq}.txt', 10)
        for frame in range(1, info_by_seq[seq]['validation_frames'] + 1):
            g = gt[gt[:, 0] == frame]
            f = full[full[:, 0] == frame]
            h = high[high[:, 0] == frame]
            valid = g[(g[:, 6] == 1) & (g[:, 7] == 1) & (g[:, 8] >= .4)]
            if len(valid) and len(f):
                similarity = overlaps(valid[:, 2:6], f[:, 2:6])
                best = similarity.argmax(axis=1)
                full_iou = similarity[np.arange(len(valid)), best]
                high_iou = max_overlap(valid[:, 2:6], h[:, 2:6])
                # Low-score output supports a possible second-association recovery.
                selected = np.flatnonzero((full_iou >= .5) & (high_iou < .3) & (f[best, 6] < .6))
                for k in selected:
                    row = valid[k]; prediction = f[best[k]]
                    events.append(dict(kind='help_candidate', sequence=seq, frame=frame,
                        gt_id=int(row[1]), full_id=int(prediction[1]), x=float(row[2]),y=float(row[3]),
                        w=float(row[4]),h=float(row[5]),full_iou=float(full_iou[k]),
                        high_iou=float(high_iou[k]),score=float(prediction[6])))
            if len(f):
                scored = g[(g[:, 6] == 1) & (g[:, 7] == 1)]
                ignored = g[(g[:, 6] != 1) | (g[:, 7] != 1)]
                any_iou = max_overlap(f[:, 2:6], g[:, 2:6])
                ignore_ioa = max_overlap(f[:, 2:6], ignored[:, 2:6], use_ioa=True)
                high_iou = max_overlap(f[:, 2:6], h[:, 2:6])
                selected = np.flatnonzero((any_iou < .1) & (ignore_ioa < .5) & (high_iou < .3))
                for k in selected:
                    row = f[k]
                    events.append(dict(kind='risk_candidate', sequence=seq, frame=frame,
                        gt_id=-1,full_id=int(row[1]),x=float(row[2]),y=float(row[3]),
                        w=float(row[4]),h=float(row[5]),full_iou=float(any_iou[k]),
                        high_iou=float(high_iou[k]),score=float(row[6])))
    # Group adjacent frames involving the same GT and full-tracker ID.
    grouped = {}
    for event in events:
        key = (event['kind'],event['sequence'],event['gt_id'],event['full_id'])
        grouped.setdefault(key,[]).append(event)
    clips = []
    for key, group in grouped.items():
        group.sort(key=lambda e:e['frame'])
        segments = []
        for event in group:
            if not segments or event['frame'] != segments[-1][-1]['frame'] + 1:
                segments.append([])
            segments[-1].append(event)
        for segment in segments:
            if len(segment) < 3:
                continue
            middle = segment[len(segment)//2]
            info = info_by_seq[middle['sequence']]
            clips.append(dict(middle,start_frame=segment[0]['frame'],end_frame=segment[-1]['frame'],
                length=len(segment),original_frame=info['first_original_frame']+middle['frame']-1))
    return sorted(clips,key=lambda e:(e['kind'],-e['length'],e['sequence'],e['frame']))


In [ ]:
seq='MOT17-02-FRCNN'
gt=load_rows(ROOT/'gt'/seq/'gt/gt.txt',9)
tables_for_check={v:load_rows(ROOT/'trackers'/v/'data'/f'{seq}.txt',10) for v in ['high_only','full_bytetrack']}
timeline=[]
for frame in range(47,67):
    person=gt[(gt[:,0]==frame)&(gt[:,1]==32)&(gt[:,6]==1)&(gt[:,7]==1)]
    assert len(person)==1
    row=dict(validation_frame=frame,original_frame=info_by_seq[seq]['first_original_frame']+frame-1,gt_id=32)
    for variant,data in tables_for_check.items():
        tracks=data[data[:,0]==frame]
        ious=overlaps(person[:,2:6],tracks[:,2:6])[0]
        k=int(ious.argmax())
        assert ious[k]>=.6, 'Poor overlap: inspect selection before interpreting IDs.'
        row[variant+'_id']=int(tracks[k,1])
        row[variant+'_iou']=float(ious[k])
    timeline.append(row)
assert all(r['high_only_id']==26 for r in timeline)
assert all(r['full_bytetrack_id']==(26 if r['original_frame']<=351 or r['original_frame']>=363 else 36) for r in timeline)
pd.DataFrame(timeline).to_csv(ROOT/'identity_timeline.csv',index=False)
display(pd.DataFrame(timeline).round(3))
frame=56
person=gt[(gt[:,0]==frame)&(gt[:,1]==32)][0]
track=tables_for_check['full_bytetrack'][(tables_for_check['full_bytetrack'][:,0]==frame)&(tables_for_check['full_bytetrack'][:,1]==36)][0]
e=dict(kind='identity_change_candidate',sequence=seq,frame=frame,gt_id=32,full_id=36,
       original_frame=info_by_seq[seq]['first_original_frame']+frame-1,
       x=float(person[2]),y=float(person[3]),w=float(person[4]),h=float(person[5]),
       score=float(track[6]),start_frame=49,end_frame=64,length=16,
       key_frames=[49,56,64],expected_full_ids=[26,36,26],expected_high_ids=[26,26,26])
selected=[e]
(ROOT/'selected_examples.json').write_text(json.dumps(selected,indent=2))
print('Three keyframes: original 350 (before), 357 (different ID), 365 (returned ID).')


## 4. Create slow-motion side-by-side clips
Left: high-confidence only. Right: full ByteTrack. Orange boxes and T labels show tracker outputs. Green boxes and G labels show scored pedestrian annotations. Gray boxes show ignored/distractor annotations. A yellow cross marks the subject's middle-frame location for attention; they are not a detector output.

The example produces a cropped detail clip and a wider context clip. Playback is 10 FPS, one-third of the original 30 FPS speed. Context includes approximately one second before and after the candidate interval. Numeric tracker IDs across panels do not need to agree. Follow ID consistency within each panel.


In [ ]:
OUTPUT = ROOT/'visual_outputs'
OUTPUT.mkdir(exist_ok=True)
tables={}
for seq in SEQUENCES:
    tables[seq]={'gt':load_rows(ROOT/'gt'/seq/'gt/gt.txt',9)}
    for variant in ['high_only','full_bytetrack']:
        tables[seq][variant]=load_rows(ROOT/'trackers'/variant/'data'/f'{seq}.txt',10)

ORANGE=(0,165,255); GREEN=(0,255,0); GRAY=(150,150,150); YELLOW=(0,255,255)

def clip_crop(e):
    i=info_by_seq[e['sequence']]
    cx=e['x']+e['w']/2;cy=e['y']+e['h']/2
    halfw=max(120,e['w']*1.8);halfh=max(140,e['h']*1.3)
    return (max(0,int(cx-halfw)),max(0,int(cy-halfh)),min(i['width'],int(cx+halfw)),min(i['height'],int(cy+halfh)))

def draw_panel(image,seq,frame,variant,crop,panel_w,e):
    x0,y0,x1,y1=crop
    panel=image[y0:y1,x0:x1].copy()
    # Both panels share the same crop and scale.
    def box(row,color,label,thickness=1):
        x,y,w,h=row[2:6]
        if x+w<x0 or y+h<y0 or x>x1 or y>y1:return
        a=(round(x-x0),round(y-y0));b=(round(x+w-x0),round(y+h-y0))
        cv2.rectangle(panel,a,b,color,thickness)
        text_at=(max(1,min(a[0],panel.shape[1]-40)),max(12,min(a[1]-3,panel.shape[0]-4)))
        cv2.putText(panel,label,text_at,cv2.FONT_HERSHEY_SIMPLEX,.34,color,1,cv2.LINE_AA)
    gt=tables[seq]['gt']; gt=gt[gt[:,0]==frame]
    for row in gt:
        scored=row[6]==1 and row[7]==1
        box(row,GREEN if scored else GRAY,f'G{int(row[1])}' if scored else f'Ignore {int(row[1])}')
    tracks=tables[seq][variant]; tracks=tracks[tracks[:,0]==frame]
    for row in tracks:box(row,ORANGE,f'T{int(row[1])} {row[6]:.2f}',2)
    # A small yellow cross indicates the selected subject's reference location.
    cx=round(e['x']+e['w']/2-x0);cy=round(e['y']+e['h']/2-y0)
    if 0<=cx<panel.shape[1] and 0<=cy<panel.shape[0]:
        cv2.drawMarker(panel,(cx,cy),YELLOW,cv2.MARKER_CROSS,12,1)
    panel_h=round(panel.shape[0]*panel_w/panel.shape[1]);panel_h+=panel_h%2
    panel=cv2.resize(panel,(panel_w,panel_h))
    header=np.zeros((66,panel_w,3),np.uint8)
    title='High-confidence only' if variant=='high_only' else 'Full ByteTrack'
    original=info_by_seq[seq]['first_original_frame']+frame-1
    cv2.putText(header,title,(12,24),cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2)
    cv2.putText(header,f'{seq}  original {original}  validation {frame}  1/3 speed',(12,48),cv2.FONT_HERSHEY_SIMPLEX,.39,(255,255,255),1)
    return np.vstack([header,panel])

def render_pair(e,frame,crop,panel_w):
    seq=e['sequence'];original=info_by_seq[seq]['first_original_frame']+frame-1
    image=cv2.imread(str(image_folders[seq]/f'{original:06d}.jpg'))
    assert image is not None
    return np.hstack([draw_panel(image,seq,frame,v,crop,panel_w,e) for v in ['high_only','full_bytetrack']])

outputs=[]
for e in selected:
    seq=e['sequence'];info=info_by_seq[seq]
    start=max(1,e['start_frame']-30);end=min(info['validation_frames'],e['end_frame']+30)
    tag=f"{e['kind']}_{seq}_original_{e['original_frame']}"
    crops={'detail':clip_crop(e),'context':(0,0,info['width'],info['height'])}
    for view,crop in crops.items():
        panel_w=640 if view=='detail' else 960
        first=render_pair(e,start,crop,panel_w)
        raw=OUTPUT/f'{tag}_{view}_raw.mp4';target=OUTPUT/f'{tag}_{view}.mp4'
        writer=cv2.VideoWriter(str(raw),cv2.VideoWriter_fourcc(*'mp4v'),10,(first.shape[1],first.shape[0]))
        assert writer.isOpened(), 'Video writer could not open.'
        try:
            for frame in range(start,end+1):writer.write(render_pair(e,frame,crop,panel_w))
        finally:writer.release()
        subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(raw),'-an','-c:v','libx264','-pix_fmt','yuv420p','-movflags','+faststart',str(target)],check=True)
        raw.unlink()
        if view=='detail':
            for frame in e['key_frames']:
                png=OUTPUT/f'{tag}_validation_{frame:04d}.png'
                assert cv2.imwrite(str(png),render_pair(e,frame,crop,panel_w))
        outputs.append(dict(kind=e['kind'],sequence=seq,view=view,path=str(target),
            validation_start=start,validation_end=end,playback_fps=10,original_fps=30,crop=crop))
    print('Created:',tag)
(ROOT/'render_manifest.json').write_text(json.dumps(outputs,indent=2))


## 5. Watch the ID changes

Watch the same G32 pedestrian on both sides. Orange T labels are tracker IDs; green G32 is the dataset's person identity.

- Before: both panels use T26 on G32.
- Middle: full ByteTrack uses T36, while high_only keeps T26.
- After: full ByteTrack returns to T26.

The PNG keyframes are original frames 350, 357 and 365. Inspect these alongside the clip and timeline. This illustrates a local identity-continuity failure if the boxes stay on the same person, even though full ByteTrack performs better overall. It does not establish that the second-stage matcher itself made the error on the exact transition frame; the changed track history can affect later high-confidence matching too.


In [ ]:
for kind in ['identity_change_candidate']:
    result=next(o for o in outputs if o['kind']==kind and o['view']=='detail')
    print(kind, result['sequence'], 'validation frames',result['validation_start'],'to',result['validation_end'])
    display(Video(result['path'],embed=True,width=1100))
print('Orange = tracker prediction. Green = scored annotation. Gray = ignored annotation. Yellow cross = attention marker.')


## 6. Download the visual evidence

Download ByteTrack_identity_change_example.zip. It contains the cropped and context videos, three before/middle/after PNGs, the measured ID timeline, source-image checks and provenance.

Send the ZIP for visual review. This replacement example avoids relying on missing ground-truth annotations. Do not change thresholds or rerun model inference for this inspection.


In [ ]:
note='Identity-continuity visual inspection: MOT17-02 pedestrian G32.\nLeft: high-confidence only. Right: full ByteTrack.\nOrange T: tracker ID. Green G: scored annotation. Gray: ignored annotation.\nFull best-overlap IDs: T26 through frame351, T36 at352-362, T26 from363.\nHigh_only best-overlap ID: T26 throughout original348-367.\nInspect original350,357,365 and the full clip before interpreting.\nThis is a strong-overlap diagnostic timeline, not an independent replay of TrackEval temporal event labels.\nClips play at one-third speed. The yellow cross is an attention marker.\n'
(ROOT/'HOW_TO_READ.txt').write_text(note)
provenance=dict(source_zip_sha256=sha256_file(SOURCE_ZIP),
    source_experiment='Official ByteTrack controlled comparison',
    no_inference_rerun=True, selected_sequences=SEQUENCES,
    screening='Strong-overlap G32 ID timeline for human inspection; not independently replayed TrackEval IDSW events',
    opencv=cv2.__version__,numpy=np.__version__)
(ROOT/'visual_provenance.json').write_text(json.dumps(provenance,indent=2))
keep=['identity_timeline.csv','selected_examples.json','image_checks.json','render_manifest.json','visual_provenance.json','HOW_TO_READ.txt']
archive_path=Path('/content/ByteTrack_identity_change_example.zip')
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for name in keep:archive.write(ROOT/name,name)
    for path in OUTPUT.iterdir():
        if path.suffix in ['.mp4','.png']:archive.write(path,Path('visual_outputs')/path.name)
print('Download:',archive_path)
files.download(str(archive_path))
